In [1]:
!pip install roboflow ultralytics opencv-python -q
print("✅ Done installing")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 145.1 MB/s eta 0:00:00
✅ Done installing


In [3]:
from roboflow import Roboflow

rf      = Roboflow(api_key="8oLSnnnn7cv17Hnmloth")
project = rf.workspace("object-detection-vpvcm").project("food-images-2lpjg")
dataset = project.version(1).download("yolov8")

print("✅ Dataset ready!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Food-Images-1 in yolov8:: 100%|██████████| 9700/9700 [00:01<00:00, 5650.91it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset ready!


In [4]:
import os
ds_path = "Food-Images-1"
print("Train images:", len(os.listdir(f"{ds_path}/train/images")))
print("Val images:  ", len(os.listdir(f"{ds_path}/valid/images")))
print("✅ Structure looks good!")

Train images: 4113
Val images:   424
✅ Structure looks good!


In [5]:
from ultralytics import YOLO

model   = YOLO("yolov8n.pt")
results = model.train(
    data    = "Food-Images-1/data.yaml",
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    name    = "food_detector",
    patience= 10,
    augment = True,
    device  = 0        # GPU
)
print("✅ Training complete!")

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Food-Images-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, pe

In [6]:
metrics = model.val()
print(f"mAP50   : {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print("Aim for mAP50 > 0.60 for good detection")

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,017,153 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1474.6±658.1 MB/s, size: 50.3 KB)
val: Scanning /content/Food-Images-1/valid/labels.cache... 424 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 424/424 148.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 3.8it/s 7.0s
                   all        424        667      0.606      0.577      0.602      0.427
             aloo gobi         29         29      0.581      0.552      0.525      0.279
            aloo matar          3          3      0.496      0.667      0.747      0.523
            aloo methi          5          5          1      0.791      0.962      0.669
                 apple          3         25      0.879      0.584       0.67      0.277
         bhindi masala          5        

In [7]:
from google.colab import files
files.download("runs/detect/food_detector/weights/best.pt")
print("✅ best.pt downloaded — save it in Model1_FoodDetection/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ best.pt downloaded — save it in Model1_FoodDetection/
